# Notebook 1: Baseline Transformer, learning curve and back-translation
WMT26 Indic MT, low-resource NMT project. Runs on **Kaggle 2×T4** (two jobs in parallel, fp16) or **Colab A100** (sequential, bf16).

Estimated time: about 5 h on 2×T4, about 2.5 h on A100.

**How to run**

1. On your laptop: `python scripts/make_bundle.py` creates `dist/nmt_bundle.zip`.
2. **Kaggle:** New Notebook, then File, then Import Notebook (this file). Next, Add Input, then Upload Dataset (`nmt_bundle.zip`).
   Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Click **Save Version, then Save & Run All (Commit)** so it
   keeps running after you close the tab. When it finishes, download `nb1_results.zip` (and `_checkpoints.zip`) from Output.
3. **Colab:** Upload this notebook, put `nmt_bundle.zip` in *My Drive* (root), Runtime, then **A100 GPU**, then Run all.
   Outputs are written straight to `MyDrive/nmt_outputs/`, so if Colab disconnects, just run all again and finished runs are skipped.
4. On your laptop, extract the results zip in the project root, then `bash run.sh eval`.

In [1]:
# ---- 1. Locate the project bundle and unpack it (works on Kaggle and Colab) ----
import os, sys, glob, zipfile, shutil, subprocess
IN_KAGGLE = os.path.exists("/kaggle")
IN_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working/project" if IN_KAGGLE else "/content/project"

def find_bundle():
    pats = ["/kaggle/input/**/nmt_bundle.zip", "/content/nmt_bundle.zip",
            "/content/drive/MyDrive/nmt_bundle.zip", "/content/drive/MyDrive/**/nmt_bundle.zip"]
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits:
            return ("zip", hits[0])
    # Kaggle auto-extracts uploaded zips -> look for an extracted project folder
    for d in glob.glob("/kaggle/input/**/src/models", recursive=True):
        return ("dir", os.path.dirname(os.path.dirname(d)))
    return (None, None)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

kind, path = find_bundle()
if kind is None and IN_COLAB:
    from google.colab import files
    print("Upload dist/nmt_bundle.zip"); up = files.upload(); kind, path = "zip", "/content/" + list(up)[0]
assert kind, "nmt_bundle.zip not found - attach it as a Kaggle dataset / put it in MyDrive"
if not os.path.exists(WORK + "/src"):
    os.makedirs(WORK, exist_ok=True)
    if kind == "zip":
        zipfile.ZipFile(path).extractall(WORK)
    else:
        shutil.copytree(path, WORK, dirs_exist_ok=True)
os.chdir(WORK); sys.path.insert(0, WORK)

# On Colab keep outputs on Google Drive so a disconnect never loses finished runs
if IN_COLAB:
    drive_out = "/content/drive/MyDrive/nmt_outputs"
    os.makedirs(drive_out, exist_ok=True)
    if not os.path.islink("outputs"):
        if os.path.isdir("outputs"):
            shutil.copytree("outputs", drive_out, dirs_exist_ok=True); shutil.rmtree("outputs")
        os.symlink(drive_out, "outputs")
print("bundle:", path, "| workdir:", os.getcwd())

Mounted at /content/drive
bundle: /content/nmt_bundle.zip | workdir: /kaggle/working/project


In [2]:
# ---- 2. Dependencies + GPU check ----
!pip install -q sentencepiece==0.2.3 sacrebleu==2.6.0 pyyaml
import torch
n = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
print("bf16 supported:", torch.cuda.is_bf16_supported() if n else None)
assert n > 0, "No GPU! Kaggle: Settings -> Accelerator -> GPU T4 x2.  Colab: Runtime -> Change runtime type -> A100"

torch 2.14.1+cu130 | GPUs: 1 ['NVIDIA A100-SXM4-40GB']
bf16 supported: True


In [3]:
# ---- helpers: skip finished runs (safe to re-run the notebook after a disconnect) ----
import json
from pathlib import Path
from src.run_queue import run_jobs

def finished(model, direction):
    f = Path("outputs") / model / direction / "run_config.json"
    return f.exists() and json.loads(f.read_text()).get("finished", False)

def todo(jobs):
    return [(n, a) for (n, a, m, d) in jobs if not finished(m, d)]

DIRECTIONS = ["en-as", "as-en", "en-lus", "lus-en", "en-mni", "mni-en", "en-bodo", "bodo-en"]  # big pairs first

In [4]:
# ---- 3. Phase A: baseline Transformer (8 directions) + learning curve (en-as, as-en at 10/25/50%) ----
jobs = [(f"baseline_{d}", ["-m", "src.models.baseline.train", "--direction", d], "baseline", d) for d in DIRECTIONS]
for frac in (0.1, 0.25, 0.5):
    for d in ("en-as", "as-en"):
        m = f"baseline_frac{frac}"
        jobs.append((f"{m}_{d}", ["-m", "src.models.baseline.train", "--direction", d, "--train-frac", str(frac),
                                  "--model-name", m], m, d))
print(run_jobs(todo(jobs)))

[   0.0 min] GPU0 START baseline_en-as
[  16.3 min] GPU0 END   baseline_en-as OK (16.3 min)
[  16.3 min] GPU0 START baseline_as-en
[  32.0 min] GPU0 END   baseline_as-en OK (15.8 min)
[  32.0 min] GPU0 START baseline_en-lus
[  48.0 min] GPU0 END   baseline_en-lus OK (16.0 min)
[  48.0 min] GPU0 START baseline_lus-en
[  64.0 min] GPU0 END   baseline_lus-en OK (16.0 min)
[  64.0 min] GPU0 START baseline_en-mni
[  77.3 min] GPU0 END   baseline_en-mni OK (13.3 min)
[  77.3 min] GPU0 START baseline_mni-en
[  90.5 min] GPU0 END   baseline_mni-en OK (13.3 min)
[  90.5 min] GPU0 START baseline_en-bodo
[  97.8 min] GPU0 END   baseline_en-bodo OK (7.3 min)
[  97.8 min] GPU0 START baseline_bodo-en
[ 105.0 min] GPU0 END   baseline_bodo-en OK (7.3 min)
[ 105.0 min] GPU0 START baseline_frac0.1_en-as
[ 109.3 min] GPU0 END   baseline_frac0.1_en-as OK (4.3 min)
[ 109.3 min] GPU0 START baseline_frac0.1_as-en
[ 113.5 min] GPU0 END   baseline_frac0.1_as-en OK (4.3 min)
[ 113.5 min] GPU0 START baseline_fra

In [5]:
# ---- 4. Phase B: synthetic data with the baseline models (one corpus per pair) ----
gen_jobs = []
for pair in ("en-as", "en-lus", "en-mni", "en-bodo"):
    if not Path(f"outputs/synthetic/{pair}/quality.json").exists():
        gen_jobs.append((f"synthetic_{pair}", ["-m", "src.models.advanced.backtranslation", "generate", "--pair", pair]))
print(run_jobs(gen_jobs))
for pair in ("en-as", "en-lus", "en-mni", "en-bodo"):
    q = json.loads(Path(f"outputs/synthetic/{pair}/quality.json").read_text())
    print(pair, {k: q[k] for k in ("n_synthetic", "synthetic_ratio", "roundtrip_chrf++")})

[   0.0 min] GPU0 START synthetic_en-as
[   2.5 min] GPU0 END   synthetic_en-as OK (2.5 min)
[   2.5 min] GPU0 START synthetic_en-lus
[   5.0 min] GPU0 END   synthetic_en-lus OK (2.5 min)
[   5.0 min] GPU0 START synthetic_en-mni
[   6.3 min] GPU0 END   synthetic_en-mni OK (1.3 min)
[   6.3 min] GPU0 START synthetic_en-bodo
[   7.0 min] GPU0 END   synthetic_en-bodo OK (0.8 min)
{'synthetic_en-as': 0, 'synthetic_en-lus': 0, 'synthetic_en-mni': 0, 'synthetic_en-bodo': 0}
en-as {'n_synthetic': 48521, 'synthetic_ratio': 1.0, 'roundtrip_chrf++': 26.43}
en-lus {'n_synthetic': 46369, 'synthetic_ratio': 1.0, 'roundtrip_chrf++': 26.77}
en-mni {'n_synthetic': 20759, 'synthetic_ratio': 1.0, 'roundtrip_chrf++': 19.55}
en-bodo {'n_synthetic': 11272, 'synthetic_ratio': 1.0, 'roundtrip_chrf++': 25.4}


In [6]:
# ---- 5. Phase C: retrain with real + synthetic data (BT for X->en, forward-translation for en->X) ----
jobs = [(f"backtranslation_{d}", ["-m", "src.models.advanced.backtranslation", "train", "--direction", d],
         "backtranslation", d) for d in DIRECTIONS]
print(run_jobs(todo(jobs)))

[   0.0 min] GPU0 START backtranslation_en-as
[  16.0 min] GPU0 END   backtranslation_en-as OK (16.0 min)
[  16.0 min] GPU0 START backtranslation_as-en
[  31.5 min] GPU0 END   backtranslation_as-en OK (15.5 min)
[  31.5 min] GPU0 START backtranslation_en-lus
[  47.8 min] GPU0 END   backtranslation_en-lus OK (16.3 min)
[  47.8 min] GPU0 START backtranslation_lus-en
[  63.3 min] GPU0 END   backtranslation_lus-en OK (15.5 min)
[  63.3 min] GPU0 START backtranslation_en-mni
[  78.8 min] GPU0 END   backtranslation_en-mni OK (15.5 min)
[  78.8 min] GPU0 START backtranslation_mni-en
[  94.3 min] GPU0 END   backtranslation_mni-en OK (15.5 min)
[  94.3 min] GPU0 START backtranslation_en-bodo
[ 108.0 min] GPU0 END   backtranslation_en-bodo OK (13.8 min)
[ 108.0 min] GPU0 START backtranslation_bodo-en
[ 122.0 min] GPU0 END   backtranslation_bodo-en OK (14.0 min)
{'backtranslation_en-as': 0, 'backtranslation_as-en': 0, 'backtranslation_en-lus': 0, 'backtranslation_lus-en': 0, 'backtranslation_en-m

In [7]:
# ---- 6. Quick dev summary ----
for m in ["baseline", "backtranslation"]:
    for d in DIRECTIONS:
        f = Path("outputs") / m / d / "run_config.json"
        if f.exists():
            r = json.loads(f.read_text())
            print(f"{m:16s} {d:8s} dev chrF++ (beam) {r.get('dev_chrf_beam', float('nan')):6.2f}  "
                  f"steps {r.get('steps')}  {r['wall_clock_s']/60:.0f} min")

baseline         en-as    dev chrF++ (beam)  29.22  steps 20000  16 min
baseline         as-en    dev chrF++ (beam)  32.36  steps 20000  16 min
baseline         en-lus   dev chrF++ (beam)  39.43  steps 20000  16 min
baseline         lus-en   dev chrF++ (beam)  37.13  steps 20000  16 min
baseline         en-mni   dev chrF++ (beam)  40.29  steps 16800  13 min
baseline         mni-en   dev chrF++ (beam)  41.89  steps 16800  13 min
baseline         en-bodo  dev chrF++ (beam)  50.66  steps 9360  7 min
baseline         bodo-en  dev chrF++ (beam)  47.30  steps 9360  7 min
backtranslation  en-as    dev chrF++ (beam)  28.00  steps 20000  16 min
backtranslation  as-en    dev chrF++ (beam)  32.62  steps 20000  15 min
backtranslation  en-lus   dev chrF++ (beam)  38.25  steps 20000  16 min
backtranslation  lus-en   dev chrF++ (beam)  35.86  steps 20000  15 min
backtranslation  en-mni   dev chrF++ (beam)  40.40  steps 20000  15 min
backtranslation  mni-en   dev chrF++ (beam)  41.65  steps 20000  15 

In [8]:
# ---- COMET (optional; installed last so it cannot break the training environment) ----
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet"], check=True)
    subprocess.run([sys.executable, "-m", "src.eval.comet_score", "--models"] + ["baseline", "backtranslation"], check=True)
except Exception as e:
    print("COMET failed (non-fatal; chrF++/BLEU are computed locally):", e)

COMET failed (non-fatal; chrF++/BLEU are computed locally): Command '['/usr/bin/python3', '-m', 'src.eval.comet_score', '--models', 'baseline', 'backtranslation']' returned non-zero exit status 1.


In [9]:
# ---- Package results: small results zip + separate checkpoints zip ----
import zipfile, os
from pathlib import Path
out_dir = "/kaggle/working" if IN_KAGGLE else "/content/drive/MyDrive"
res = f"{out_dir}/nb1_results.zip"; ck = f"{out_dir}/nb1_checkpoints.zip"
with zipfile.ZipFile(res, "w", zipfile.ZIP_DEFLATED) as zr, zipfile.ZipFile(ck, "w") as zc:
    for root in ["baseline", "baseline_frac0.1", "baseline_frac0.25", "baseline_frac0.5", "backtranslation", "synthetic"]:
        for f in Path("outputs", root).rglob("*") if Path("outputs", root).exists() else []:
            if f.is_file():
                (zc if f.suffix == ".pt" or "checkpoint" in f.parts else zr).write(f, f.as_posix())
    for f in Path("outputs/logs").glob("*.log"):
        zr.write(f, f.as_posix())
for z in (res, ck):
    print(z, round(os.path.getsize(z) / 1e6, 1), "MB")
print("Download the zip(s) and extract them in the project root (they contain outputs/...).")

/kaggle/working/nb1_results.zip 10.7 MB
/kaggle/working/nb1_checkpoints.zip 822.1 MB
Download the zip(s) and extract them in the project root (they contain outputs/...).
